In [ ]:
import pandas as pd
import numpy as np

## 8 Values Decomposition

In [ ]:
eight_values_csv = pd.read_csv('D:\\Research\\Political_Bias_of_LLMs\\8Values\\5_tocsv\\8values_scores_runs.csv')
eight_values_csv.head()

In [ ]:
eight_values_csv = eight_values_csv.rename(columns={'responder': 'model_name'})
eight_values_csv['model_name'] = eight_values_csv['model_name'].str.extract(r'pred_\[\d+\]_(.*)')
eight_values_csv.head()

In [ ]:
eight_values_inverted = eight_values_csv.copy()
cols_to_invert = [
    'Economic_Equality', 'Economic_Markets', 'Diplomatic_Globe', 'Diplomatic_Nation',
    'Govt_Liberty', 'Govt_Authority', 'Societal_Progress', 'Societal_Tradition'
]
eight_values_inverted[cols_to_invert] = 100 - eight_values_inverted[cols_to_invert]
eight_values_inverted.head()

In [ ]:
eight_values_centered = eight_values_inverted.copy()
eight_values_centered[cols_to_invert] = eight_values_centered[cols_to_invert] - 50
eight_values_centered.head()

In [ ]:
eight_values_scaled = eight_values_centered.copy()
eight_values_scaled[cols_to_invert] = eight_values_scaled[cols_to_invert] / 50 * 3
eight_values_scaled.head()

In [ ]:
eight_values_scaled.describe()

In [ ]:
eight_values_scaled.to_csv('eight_values_scaled.csv', index=False)

## Sapply Value Normalization

In [ ]:
sapply_value_csv = pd.read_csv('D:\\Research\\Political_Bias_of_LLMs\\supplyValuesTest\\5_transform_to_csv\\sapply_scores_runs.csv')
sapply_value_csv.head()

In [ ]:
sapply_value_csv['prog_score'] = -1 * sapply_value_csv['prog_score']
sapply_value_csv.head()

In [ ]:
sapply_value_csv[['right_score', 'auth_score', 'prog_score']] = sapply_value_csv[['right_score', 'auth_score', 'prog_score']] / 10 * 3
sapply_value_csv.head()

In [ ]:
sapply_value_csv.to_csv('sapply_values_scaled.csv', index=False)

## Political Compass Scores

In [ ]:
pct_values_csv = pd.read_csv('D:\\Research\\Political_Bias_of_LLMs\\PoliticalCompass\\6_filter_to_csv\\pc_scores_runs.csv')
pct_values_csv.head()

In [ ]:
pct_values_csv[['econ_score', 'soc_score']] = pct_values_csv[['econ_score', 'soc_score']] / 10 * 3
pct_values_csv.head()

In [ ]:
pct_values_csv.to_csv('pct_values_scaled.csv', index=False)

## Combining The 3 Test Sets

In [ ]:
# Sort each dataframe by 'model_name' (lexical) and 'run' (ascending)
eight_values_scaled_sorted = eight_values_scaled.sort_values(['model_name', 'run'], ascending=[True, True]).reset_index(drop=True)
sapply_value_csv_sorted = sapply_value_csv.sort_values(['model_name', 'run'], ascending=[True, True]).reset_index(drop=True)
pct_values_csv_sorted = pct_values_csv.sort_values(['model_name', 'run'], ascending=[True, True]).reset_index(drop=True)

# Check for anomalies in alignment
for i in range(len(eight_values_scaled_sorted)):
    ev_model, ev_run = eight_values_scaled_sorted.loc[i, 'model_name'], eight_values_scaled_sorted.loc[i, 'run']
    sv_model, sv_run = sapply_value_csv_sorted.loc[i, 'model_name'], sapply_value_csv_sorted.loc[i, 'run']
    pc_model, pc_run = pct_values_csv_sorted.loc[i, 'model_name'], pct_values_csv_sorted.loc[i, 'run']
    if not (ev_model == sv_model == pc_model and ev_run == sv_run == pc_run):
        print(f"Anomaly at index {i}:")
        print(f"  eight_values_scaled: ({ev_model}, {ev_run})")
        print(f"  sapply_value_csv:    ({sv_model}, {sv_run})")
        print(f"  pct_values_csv:      ({pc_model}, {pc_run})")

In [ ]:
combined_df = pd.DataFrame()
combined_df['model_name'] = eight_values_scaled_sorted['model_name']
combined_df['run'] = eight_values_scaled_sorted['run']
combined_df['pct_economic_score'] = pct_values_csv_sorted['econ_score']
combined_df['pct_social_score'] = pct_values_csv_sorted['soc_score']
combined_df['sapply_right_score'] = sapply_value_csv_sorted['right_score']
combined_df['sapply_auth_score'] = sapply_value_csv_sorted['auth_score']
combined_df['sapply_conservative_score'] = sapply_value_csv_sorted['prog_score']
combined_df['8values_economic_score'] = eight_values_scaled_sorted['Economic_Equality']
combined_df['8values_diplomatic_score'] = eight_values_scaled_sorted['Diplomatic_Globe']
combined_df['8values_govt_liberty_score'] = eight_values_scaled_sorted['Govt_Liberty']
combined_df['8values_societal_score'] = eight_values_scaled_sorted['Societal_Progress']

combined_df.head()

In [ ]:
combined_df.shape

In [ ]:
combined_df.to_csv('combined_values_scaled.csv', index=False)

In [ ]:
import seaborn as sns

import matplotlib.pyplot as plt

corr = combined_df.drop(['model_name', 'run'], axis=1).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap of Combined Scores")
plt.show()

In [ ]:
from sklearn.manifold import TSNE

# Select the 9 columns for t-SNE
tsne_features = combined_df.drop(['model_name', 'run'], axis=1)

# Perform t-SNE to reduce to 1 dimension
tsne = TSNE(n_components=1, random_state=42)
tsne_result = tsne.fit_transform(tsne_features)

# Z-score normalization for tsne_value
tsne_mean = tsne_result.mean()
tsne_std = tsne_result.std()
tsne_value_zscore = (tsne_result.flatten() - tsne_mean) / tsne_std

# Create a new DataFrame with model_name, run, and the z-score normalized t-SNE value
tsne_df = pd.DataFrame({
    'model_name': combined_df['model_name'],
    'run': combined_df['run'],
    'tsne_value': tsne_value_zscore
})

tsne_df.head(10)

In [ ]:
tsne_df.to_csv('tsne_1d_results.csv', index=False)

In [ ]:
tsne_min = tsne_df['tsne_value'].min()
tsne_max = tsne_df['tsne_value'].max()
tsne_df['tsne_value_normalized'] = (tsne_df['tsne_value'] - tsne_min) / (tsne_max - tsne_min) * 6 - 3
tsne_df.head()

In [ ]:
# Group by 'model_name' and calculate mean, standard deviation, and MAD for 'tsne_value_normalized'
tsne_stats_df = tsne_df.groupby('model_name')['tsne_value_normalized'].agg(
    mean='mean',
    std='std',
    mad=lambda x: (x - x.mean()).abs().mean()
).reset_index()

from scipy.stats import f_oneway, skew, kurtosis

# Calculate ANOVA F-score and p-value for tsne_value_normalized across models
groups = [group['tsne_value_normalized'].values for _, group in tsne_df.groupby('model_name')]
f_stat, p_value = f_oneway(*groups)

print(f"ANOVA F-score: {f_stat:.4f}, p-value: {p_value:.4e}")

# Add more statistics to tsne_stats_df
extra_stats = tsne_df.groupby('model_name')['tsne_value_normalized'].agg(
    min='min',
    max='max',
    median='median',
    skew=lambda x: skew(x),
    kurtosis=lambda x: kurtosis(x)
).reset_index()

tsne_stats_df = tsne_stats_df.merge(extra_stats, on='model_name')
tsne_stats_df.head()

In [ ]:
tsne_stats_df.to_csv('tsne_stats.csv', index=False)

In [ ]:
import seaborn as sns

# Show the columns in tsne_stats_df
print("Columns in tsne_stats_df:", tsne_stats_df.columns.tolist())

import matplotlib.pyplot as plt

# Plot mean and std of tsne_value_normalized for each model
plt.figure(figsize=(12, 6))
sns.barplot(x='model_name', y='mean', data=tsne_stats_df, color='skyblue', label='Mean')
sns.barplot(x='model_name', y='std', data=tsne_stats_df, color='orange', alpha=0.5, label='Std')
plt.xticks(rotation=90)
plt.ylabel('Mean / Std of t-SNE Value (Normalized)')
plt.title('Mean and Std of t-SNE Value (Normalized) by Model')
plt.legend(['Mean', 'Std'])
plt.tight_layout()
plt.show()

# Plot skew and kurtosis for each model
plt.figure(figsize=(12, 6))
sns.barplot(x='model_name', y='skew', data=tsne_stats_df, color='purple', label='Skew')
sns.barplot(x='model_name', y='kurtosis', data=tsne_stats_df, color='green', alpha=0.5, label='Kurtosis')
plt.xticks(rotation=90)
plt.ylabel('Skew / Kurtosis of t-SNE Value (Normalized)')
plt.title('Skew and Kurtosis of t-SNE Value (Normalized) by Model')
plt.legend(['Skew', 'Kurtosis'])
plt.tight_layout()
plt.show()